In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a 2D max pooling operation for image/feature map downsampling.
  The program should take an input tensor and produce an output tensor by applying max pooling with specified kernel size, stride, and padding.
</p>

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 420 180" style="display:block; margin:20px auto;" width="420" height="180" font-family="monospace">
  <rect width="420" height="180" rx="8" fill="#222"/>
  <defs>
    <marker id="arrpool" markerWidth="8" markerHeight="6" refX="8" refY="3" orient="auto">
      <polygon points="0 0, 8 3, 0 6" fill="#ccc"/>
    </marker>
  </defs>

  <!-- Input label -->
  <text x="75" y="18" text-anchor="middle" fill="#ccc" font-size="11" font-weight="bold">Input (4x4)</text>

  <!-- Input grid background -->
  <rect x="15" y="24" width="120" height="120" rx="3" fill="#333" stroke="#555" stroke-width="0.5"/>

  <!-- Region highlights (2x2 stride=2) with dashed borders -->
  <!-- Top-left region (green) -->
  <rect x="15" y="24" width="60" height="60" fill="#1e3d2d" opacity="0.5"/>
  <rect x="15" y="24" width="60" height="60" fill="none" stroke="#44aa66" stroke-width="1.5" stroke-dasharray="4,2"/>
  <!-- Top-right region (blue) -->
  <rect x="75" y="24" width="60" height="60" fill="#1e2d3d" opacity="0.5"/>
  <rect x="75" y="24" width="60" height="60" fill="none" stroke="#4488cc" stroke-width="1.5" stroke-dasharray="4,2"/>
  <!-- Bottom-left region (amber) -->
  <rect x="15" y="84" width="60" height="60" fill="#3d2d1e" opacity="0.5"/>
  <rect x="15" y="84" width="60" height="60" fill="none" stroke="#cc8844" stroke-width="1.5" stroke-dasharray="4,2"/>
  <!-- Bottom-right region (purple) -->
  <rect x="75" y="84" width="60" height="60" fill="#3d1e3d" opacity="0.5"/>
  <rect x="75" y="84" width="60" height="60" fill="none" stroke="#aa44aa" stroke-width="1.5" stroke-dasharray="4,2"/>

  <!-- Grid lines -->
  <line x1="45" y1="24" x2="45" y2="144" stroke="#555" stroke-width="0.5"/>
  <line x1="75" y1="24" x2="75" y2="144" stroke="#555" stroke-width="0.5"/>
  <line x1="105" y1="24" x2="105" y2="144" stroke="#555" stroke-width="0.5"/>
  <line x1="15" y1="54" x2="135" y2="54" stroke="#555" stroke-width="0.5"/>
  <line x1="15" y1="84" x2="135" y2="84" stroke="#555" stroke-width="0.5"/>
  <line x1="15" y1="114" x2="135" y2="114" stroke="#555" stroke-width="0.5"/>

  <!-- Input values row 1: 1,3,2,4 -->
  <text x="30" y="43" text-anchor="middle" fill="#999" font-size="12">1</text>
  <text x="60" y="43" text-anchor="middle" fill="#999" font-size="12">3</text>
  <text x="90" y="43" text-anchor="middle" fill="#999" font-size="12">2</text>
  <text x="120" y="43" text-anchor="middle" fill="#999" font-size="12">4</text>
  <!-- Row 2: 5,8,6,7 -->
  <text x="30" y="73" text-anchor="middle" fill="#999" font-size="12">5</text>
  <text x="60" y="73" text-anchor="middle" fill="#fff" font-size="12" font-weight="bold">8</text>
  <text x="90" y="73" text-anchor="middle" fill="#999" font-size="12">6</text>
  <text x="120" y="73" text-anchor="middle" fill="#fff" font-size="12" font-weight="bold">7</text>
  <!-- Row 3: 9,2,4,3 -->
  <text x="30" y="103" text-anchor="middle" fill="#fff" font-size="12" font-weight="bold">9</text>
  <text x="60" y="103" text-anchor="middle" fill="#999" font-size="12">2</text>
  <text x="90" y="103" text-anchor="middle" fill="#999" font-size="12">4</text>
  <text x="120" y="103" text-anchor="middle" fill="#999" font-size="12">3</text>
  <!-- Row 4: 1,6,5,8 -->
  <text x="30" y="133" text-anchor="middle" fill="#999" font-size="12">1</text>
  <text x="60" y="133" text-anchor="middle" fill="#999" font-size="12">6</text>
  <text x="90" y="133" text-anchor="middle" fill="#999" font-size="12">5</text>
  <text x="120" y="133" text-anchor="middle" fill="#fff" font-size="12" font-weight="bold">8</text>

  <!-- Arrow with "max" label -->
  <text x="172" y="76" text-anchor="middle" fill="#ccc" font-size="10" font-style="italic">max</text>
  <line x1="150" y1="84" x2="195" y2="84" stroke="#ccc" stroke-width="1.5" marker-end="url(#arrpool)"/>

  <!-- Output label -->
  <text x="240" y="52" text-anchor="middle" fill="#ccc" font-size="11" font-weight="bold">Output (2x2)</text>

  <!-- Output grid background -->
  <rect x="210" y="60" width="60" height="60" rx="3" fill="#333" stroke="#555" stroke-width="0.5"/>

  <!-- Output grid lines -->
  <line x1="240" y1="60" x2="240" y2="120" stroke="#555" stroke-width="0.5"/>
  <line x1="210" y1="90" x2="270" y2="90" stroke="#555" stroke-width="0.5"/>

  <!-- Output region color coding -->
  <rect x="210" y="60" width="30" height="30" fill="#1e3d2d" opacity="0.5"/>
  <rect x="240" y="60" width="30" height="30" fill="#1e2d3d" opacity="0.5"/>
  <rect x="210" y="90" width="30" height="30" fill="#3d2d1e" opacity="0.5"/>
  <rect x="240" y="90" width="30" height="30" fill="#3d1e3d" opacity="0.5"/>

  <!-- Output values: [[8,7],[9,8]] -->
  <text x="225" y="80" text-anchor="middle" fill="#fff" font-size="13" font-weight="bold">8</text>
  <text x="255" y="80" text-anchor="middle" fill="#fff" font-size="13" font-weight="bold">7</text>
  <text x="225" y="110" text-anchor="middle" fill="#fff" font-size="13" font-weight="bold">9</text>
  <text x="255" y="110" text-anchor="middle" fill="#fff" font-size="13" font-weight="bold">8</text>

  <!-- Legend -->
  <text x="300" y="70" fill="#ccc" font-size="9">kernel: 2x2</text>
  <text x="300" y="84" fill="#ccc" font-size="9">stride: 2</text>
  <text x="300" y="98" fill="#ccc" font-size="9">padding: 0</text>

  <!-- Footer note -->
  <text x="75" y="164" text-anchor="middle" fill="#666" font-size="9">dashed borders = pooling windows</text>
</svg>

<h2>Implementation Requirements</h2>
<ul>
  <li>External libraries are not permitted</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in tensor <code>output</code></li>
</ul>

<h2>Max Pooling Operation</h2>
<p>
  For each output position (n, c, h_out, w_out), compute the maximum value over the corresponding input window:
  <br>
  <code>output[n, c, h_out, w_out] = max(input[n, c, h:h+kernel_size, w:w+kernel_size])</code>
  <br>
  where h = h_out * stride and w = w_out * stride
</p>

<h2>Example 1:</h2>
<pre>
Input:  input = [[[[1.0, 2.0, 3.0],
                   [4.0, 5.0, 6.0],
                   [7.0, 8.0, 9.0]]]]
        kernel_size = 2
        stride = 1
        padding = 0
Output: output = [[[[5.0, 6.0],
                    [8.0, 9.0]]]]
</pre>

<h2>Example 2:</h2>
<pre>
Input:  input = [[[[1.0, 2.0, 3.0, 4.0, 5.0],
                   [6.0, 7.0, 8.0, 9.0, 10.0],
                   [11.0, 12.0, 13.0, 14.0, 15.0],
                   [16.0, 17.0, 18.0, 19.0, 20.0],
                   [21.0, 22.0, 23.0, 24.0, 25.0]]]]
        kernel_size = 3
        stride = 1
        padding = 1
Output: output = [[[[7.0, 8.0, 9.0, 10.0, 10.0],
                    [12.0, 13.0, 14.0, 15.0, 15.0],
                    [17.0, 18.0, 19.0, 20.0, 20.0],
                    [22.0, 23.0, 24.0, 25.0, 25.0],
                    [22.0, 23.0, 24.0, 25.0, 25.0]]]]
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 ≤ N ≤ 100 (batch size)</li>
  <li>1 ≤ C ≤ 512 (channels)</li>
  <li>1 ≤ H, W ≤ 1024 (height, width)</li>
  <li>1 ≤ kernel_size ≤ 16</li>
  <li>1 ≤ stride ≤ 16</li>
  <li>0 ≤ padding ≤ 16</li>
  <li>Input and output tensors use float32 precision</li>

  <li>Performance is measured with <code>N</code> = 4, <code>kernel_size</code> = 3, <code>stride</code> = 2</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// input, output are device pointers (i.e. pointers to memory on the GPU)
extern "C" void solve(const float* input, float* output, int N, int C, int H, int W,
                      int kernel_size, int stride, int padding) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# input, output are tensors on the GPU
@cute.jit
def solve(
    input: cute.Tensor,
    output: cute.Tensor,
    N: cute.Int32,
    C: cute.Int32,
    H: cute.Int32,
    W: cute.Int32,
    kernel_size: cute.Int32,
    stride: cute.Int32,
    padding: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# input is a tensor on the GPU
@jax.jit
def solve(
    input: jax.Array, N: int, C: int, H: int, W: int, kernel_size: int, stride: int, padding: int
) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# input, output are device pointers (i.e. pointers to memory on the GPU)
@export
def solve(
    input: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    N: Int32,
    C: Int32,
    H: Int32,
    W: Int32,
    kernel_size: Int32,
    stride: Int32,
    padding: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# input, output are tensors on the GPU
def solve(input, output, N, C, H, W, kernel_size, stride, padding):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# input, output are tensors on the GPU
def solve(input, output, N, C, H, W, kernel_size, stride, padding):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="2D Max Pooling", atol=1e-05, rtol=1e-05, num_gpus=1, access_tier="free"
        )

    def reference_impl(
        self,
        input: torch.Tensor,
        output: torch.Tensor,
        N: int,
        C: int,
        H: int,
        W: int,
        kernel_size: int,
        stride: int,
        padding: int,
    ):
        input_tensor = input.view(N, C, H, W)

        # Apply max pooling
        result = torch.nn.functional.max_pool2d(
            input_tensor, kernel_size=kernel_size, stride=stride, padding=padding
        )

        output.copy_(result.flatten())

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "input": (ctypes.POINTER(ctypes.c_float), "in"),
            "output": (ctypes.POINTER(ctypes.c_float), "out"),
            "N": (ctypes.c_int, "in"),
            "C": (ctypes.c_int, "in"),
            "H": (ctypes.c_int, "in"),
            "W": (ctypes.c_int, "in"),
            "kernel_size": (ctypes.c_int, "in"),
            "stride": (ctypes.c_int, "in"),
            "padding": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        """Simple test case matching the example in challenge.html"""
        dtype = torch.float32
        N, C, H, W = 1, 1, 3, 3
        kernel_size, stride, padding = 2, 1, 0

        # Create input tensor: [[[[1, 2, 3], [4, 5, 6], [7, 8, 9]]]]
        input_tensor = torch.tensor(
            [[[[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]]], device="cuda", dtype=dtype
        )

        # Calculate output dimensions
        H_out = (H + 2 * padding - kernel_size) // stride + 1
        W_out = (W + 2 * padding - kernel_size) // stride + 1
        output_tensor = torch.empty(N * C * H_out * W_out, device="cuda", dtype=dtype)

        return {
            "input": input_tensor.flatten(),
            "output": output_tensor,
            "N": N,
            "C": C,
            "H": H,
            "W": W,
            "kernel_size": kernel_size,
            "stride": stride,
            "padding": padding,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """Comprehensive test suite covering various scenarios and edge cases"""
        dtype = torch.float32
        test_cases = []

        # Set seed for reproducible random tests
        torch.manual_seed(42)

        # Test case 1: 2x2 kernel, stride 2, no padding (deterministic)
        N, C, H, W = 1, 1, 4, 4
        kernel_size, stride, padding = 2, 2, 0
        H_out = (H + 2 * padding - kernel_size) // stride + 1
        W_out = (W + 2 * padding - kernel_size) // stride + 1

        input_tensor = torch.tensor(
            [
                [
                    [
                        [1.0, 2.0, 3.0, 4.0],
                        [5.0, 6.0, 7.0, 8.0],
                        [9.0, 10.0, 11.0, 12.0],
                        [13.0, 14.0, 15.0, 16.0],
                    ]
                ]
            ],
            device="cuda",
            dtype=dtype,
        )
        output_tensor = torch.empty(N * C * H_out * W_out, device="cuda", dtype=dtype)

        test_cases.append(
            {
                "input": input_tensor.flatten(),
                "output": output_tensor,
                "N": N,
                "C": C,
                "H": H,
                "W": W,
                "kernel_size": kernel_size,
                "stride": stride,
                "padding": padding,
            }
        )

        # Test case 2: 3x3 kernel, stride 1, padding 1 (random data)
        N, C, H, W = 1, 2, 5, 5
        kernel_size, stride, padding = 3, 1, 1
        H_out = (H + 2 * padding - kernel_size) // stride + 1
        W_out = (W + 2 * padding - kernel_size) // stride + 1

        input_tensor = torch.randn(N, C, H, W, device="cuda", dtype=dtype)
        output_tensor = torch.empty(N * C * H_out * W_out, device="cuda", dtype=dtype)

        test_cases.append(
            {
                "input": input_tensor.flatten(),
                "output": output_tensor,
                "N": N,
                "C": C,
                "H": H,
                "W": W,
                "kernel_size": kernel_size,
                "stride": stride,
                "padding": padding,
            }
        )

        # Test case 3: 1x1 kernel, stride 1, no padding (identity operation)
        N, C, H, W = 2, 3, 8, 8
        kernel_size, stride, padding = 1, 1, 0
        H_out = (H + 2 * padding - kernel_size) // stride + 1
        W_out = (W + 2 * padding - kernel_size) // stride + 1

        input_tensor = torch.randn(N, C, H, W, device="cuda", dtype=dtype)
        output_tensor = torch.empty(N * C * H_out * W_out, device="cuda", dtype=dtype)

        test_cases.append(
            {
                "input": input_tensor.flatten(),
                "output": output_tensor,
                "N": N,
                "C": C,
                "H": H,
                "W": W,
                "kernel_size": kernel_size,
                "stride": stride,
                "padding": padding,
            }
        )

        # Test case 4: Large kernel with padding
        N, C, H, W = 1, 1, 10, 10
        kernel_size, stride, padding = 5, 2, 2
        H_out = (H + 2 * padding - kernel_size) // stride + 1
        W_out = (W + 2 * padding - kernel_size) // stride + 1

        input_tensor = torch.randn(N, C, H, W, device="cuda", dtype=dtype)
        output_tensor = torch.empty(N * C * H_out * W_out, device="cuda", dtype=dtype)

        test_cases.append(
            {
                "input": input_tensor.flatten(),
                "output": output_tensor,
                "N": N,
                "C": C,
                "H": H,
                "W": W,
                "kernel_size": kernel_size,
                "stride": stride,
                "padding": padding,
            }
        )

        # Test case 5: Edge case with small dimensions
        N, C, H, W = 1, 1, 2, 2
        kernel_size, stride, padding = 2, 1, 0
        H_out = (H + 2 * padding - kernel_size) // stride + 1
        W_out = (W + 2 * padding - kernel_size) // stride + 1

        input_tensor = torch.tensor([[[[1.0, 2.0], [3.0, 4.0]]]], device="cuda", dtype=dtype)
        output_tensor = torch.empty(N * C * H_out * W_out, device="cuda", dtype=dtype)

        test_cases.append(
            {
                "input": input_tensor.flatten(),
                "output": output_tensor,
                "N": N,
                "C": C,
                "H": H,
                "W": W,
                "kernel_size": kernel_size,
                "stride": stride,
                "padding": padding,
            }
        )

        # Test case 6: Boundary conditions - kernel size equals input size
        N, C, H, W = 1, 1, 3, 3
        kernel_size, stride, padding = 3, 1, 0
        H_out = (H + 2 * padding - kernel_size) // stride + 1
        W_out = (W + 2 * padding - kernel_size) // stride + 1

        input_tensor = torch.tensor(
            [[[[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]]], device="cuda", dtype=dtype
        )
        output_tensor = torch.empty(N * C * H_out * W_out, device="cuda", dtype=dtype)

        test_cases.append(
            {
                "input": input_tensor.flatten(),
                "output": output_tensor,
                "N": N,
                "C": C,
                "H": H,
                "W": W,
                "kernel_size": kernel_size,
                "stride": stride,
                "padding": padding,
            }
        )

        # Test case 7: Large padding relative to input size
        N, C, H, W = 1, 1, 4, 4
        kernel_size, stride, padding = 2, 1, 1
        H_out = (H + 2 * padding - kernel_size) // stride + 1
        W_out = (W + 2 * padding - kernel_size) // stride + 1

        input_tensor = torch.tensor(
            [
                [
                    [
                        [1.0, 2.0, 3.0, 4.0],
                        [5.0, 6.0, 7.0, 8.0],
                        [9.0, 10.0, 11.0, 12.0],
                        [13.0, 14.0, 15.0, 16.0],
                    ]
                ]
            ],
            device="cuda",
            dtype=dtype,
        )
        output_tensor = torch.empty(N * C * H_out * W_out, device="cuda", dtype=dtype)

        test_cases.append(
            {
                "input": input_tensor.flatten(),
                "output": output_tensor,
                "N": N,
                "C": C,
                "H": H,
                "W": W,
                "kernel_size": kernel_size,
                "stride": stride,
                "padding": padding,
            }
        )

        # Test case 8: Multiple channels with different patterns
        N, C, H, W = 1, 3, 6, 6
        kernel_size, stride, padding = 2, 2, 1
        H_out = (H + 2 * padding - kernel_size) // stride + 1
        W_out = (W + 2 * padding - kernel_size) // stride + 1

        # Create structured input with different patterns per channel
        input_tensor = torch.zeros(N, C, H, W, device="cuda", dtype=dtype)
        input_tensor[0, 0, :, :] = torch.arange(H * W, device="cuda", dtype=dtype).reshape(H, W)
        input_tensor[0, 1, :, :] = (
            torch.arange(H * W, device="cuda", dtype=dtype).reshape(H, W).flip(0)
        )
        input_tensor[0, 2, :, :] = (
            torch.arange(H * W, device="cuda", dtype=dtype).reshape(H, W).flip(1)
        )

        output_tensor = torch.empty(N * C * H_out * W_out, device="cuda", dtype=dtype)

        test_cases.append(
            {
                "input": input_tensor.flatten(),
                "output": output_tensor,
                "N": N,
                "C": C,
                "H": H,
                "W": W,
                "kernel_size": kernel_size,
                "stride": stride,
                "padding": padding,
            }
        )

        # Test case 9: Extreme values and edge cases
        N, C, H, W = 1, 1, 5, 5
        kernel_size, stride, padding = 2, 1, 0
        H_out = (H + 2 * padding - kernel_size) // stride + 1
        W_out = (W + 2 * padding - kernel_size) // stride + 1

        # Create input with extreme values
        input_tensor = torch.tensor(
            [
                [
                    [
                        [1e6, -1e6, 0.0, 1e-6, -1e-6],
                        [float("inf"), float("-inf"), 1.0, 2.0, 3.0],
                        [4.0, 5.0, 6.0, 7.0, 8.0],
                        [9.0, 10.0, 11.0, 12.0, 13.0],
                        [14.0, 15.0, 16.0, 17.0, 18.0],
                    ]
                ]
            ],
            device="cuda",
            dtype=dtype,
        )
        output_tensor = torch.empty(N * C * H_out * W_out, device="cuda", dtype=dtype)

        test_cases.append(
            {
                "input": input_tensor.flatten(),
                "output": output_tensor,
                "N": N,
                "C": C,
                "H": H,
                "W": W,
                "kernel_size": kernel_size,
                "stride": stride,
                "padding": padding,
            }
        )

        # Test case 10: Non-power-of-two dimensions
        N, C, H, W = 1, 1, 7, 11
        kernel_size, stride, padding = 3, 2, 1
        H_out = (H + 2 * padding - kernel_size) // stride + 1
        W_out = (W + 2 * padding - kernel_size) // stride + 1

        input_tensor = torch.randn(N, C, H, W, device="cuda", dtype=dtype)
        output_tensor = torch.empty(N * C * H_out * W_out, device="cuda", dtype=dtype)

        test_cases.append(
            {
                "input": input_tensor.flatten(),
                "output": output_tensor,
                "N": N,
                "C": C,
                "H": H,
                "W": W,
                "kernel_size": kernel_size,
                "stride": stride,
                "padding": padding,
            }
        )

        return test_cases

    def generate_performance_test(self) -> Dict[str, Any]:
        """Large test case for performance evaluation"""
        dtype = torch.float32
        # Reasonable size for performance testing without memory issues
        N, C, H, W = 4, 64, 256, 256  # 4 batches, 64 channels, 256x256 spatial
        kernel_size, stride, padding = 3, 2, 1

        H_out = (H + 2 * padding - kernel_size) // stride + 1
        W_out = (W + 2 * padding - kernel_size) // stride + 1

        # Use seeded random for reproducible performance tests
        torch.manual_seed(123)
        input_tensor = torch.randn(N, C, H, W, device="cuda", dtype=dtype)
        output_tensor = torch.empty(N * C * H_out * W_out, device="cuda", dtype=dtype)

        return {
            "input": input_tensor.flatten(),
            "output": output_tensor,
            "N": N,
            "C": C,
            "H": H,
            "W": W,
            "kernel_size": kernel_size,
            "stride": stride,
            "padding": padding,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
